In [0]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
 
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)
 
S1_PATH   = "abfss://silverlayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"
S2_PATH   = "abfss://silverlayer@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/"
S3_PATH   = "abfss://silverlayer@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3/"
GOLD_PATH = "abfss://goldlayer@cryptodl.dfs.core.windows.net/crypto_gold/"
 
spark = SparkSession.builder.getOrCreate()
 

# CELL 1 — Read Silver Sources

 
log.info("Reading Silver sources...")
 
s1 = (
    spark.read.format("delta").load(S1_PATH)
    .withColumn("trade_date", F.to_date("trade_date"))
    .withColumnRenamed("Symbol", "symbol")
)
 
s2 = (
    spark.read.format("delta").load(S2_PATH)
    .withColumn("trade_date", F.to_date("trade_date"))
)
 
s3 = (
    spark.read.format("delta").load(S3_PATH)
    .withColumn("trade_date", F.to_date("trade_date"))
)
 
log.info(f"S1 rows: {s1.count():,}")
log.info(f"S2 rows: {s2.count():,}")
log.info(f"S3 rows: {s3.count():,}")
 

In [0]:

log.info("Building Gold Layer...")
 
# Aggregate S2: one sentiment score per symbol+date (avg of multiple posts)
s2_agg = (
    s2.groupBy("symbol", "trade_date")
      .agg(
          F.avg("sentiment_score").alias("avg_sentiment_score"),
          F.count("*").alias("post_count"),
          F.avg("engagement_score").alias("avg_engagement_score"),
          F.first("sentiment_label").alias("dominant_sentiment")
      )
)
 
# Aggregate S3: one metric row per symbol+date
s3_agg = (
    s3.groupBy("symbol", "trade_date")
      .agg(
          F.avg("active_addresses").cast("long").alias("active_addresses"),
          F.avg("transaction_count").cast("long").alias("transaction_count"),
          F.avg("network_fees").alias("avg_network_fees"),
          F.sum("net_exchange_flow").alias("net_exchange_flow"),
          F.first("flow_signal").alias("flow_signal")
      )
)
 
# JOIN all three
gold_df = (
    s1.join(s2_agg, on=["symbol", "trade_date"], how="left")
      .join(s3_agg, on=["symbol", "trade_date"], how="left")
)
 

In [0]:
# ── Technical Indicators 
w7  = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-6, 0)
w14 = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-13, 0)
w30 = Window.partitionBy("symbol").orderBy("trade_date").rowsBetween(-29, 0)
w_all = Window.partitionBy("symbol").orderBy("trade_date")
 
gold_enriched = (
    gold_df
    # Moving Averages
    .withColumn("ma_7",  F.avg("close_price").over(w7))
    .withColumn("ma_14", F.avg("close_price").over(w14))
    .withColumn("ma_30", F.avg("close_price").over(w30))
 
    # Daily price change %
    .withColumn("prev_close", F.lag("close_price", 1).over(w_all))
    .withColumn("price_change_pct",
        F.round(((F.col("close_price") - F.col("prev_close")) / F.col("prev_close")) * 100, 4)
    )
 
    # Price range
    .withColumn("price_range", F.col("high_price") - F.col("low_price"))
 
    # Volatility (7-day std)
    .withColumn("volatility_7d", F.stddev("close_price").over(w7))
 
    # MA signal: is price above or below 7-day MA?
    .withColumn("ma_signal",
        F.when(F.col("close_price") > F.col("ma_7"), "above_ma")
         .when(F.col("close_price") < F.col("ma_7"), "below_ma")
         .otherwise("at_ma")
    )
 
    # Composite score: sentiment + flow signal for ML
    .withColumn("bullish_score",
        F.when(F.col("dominant_sentiment") == "positive", 1)
         .when(F.col("dominant_sentiment") == "negative", -1)
         .otherwise(0)
        +
        F.when(F.col("flow_signal") == "accumulation", 1)
         .when(F.col("flow_signal") == "distribution", -1)
         .otherwise(0)
    )
 
    # Gold metadata
    .withColumn("gold_processed_at", F.current_timestamp())
 
    # Clean up temp column
    .drop("prev_close", "data_source", "ingestion_timestamp")
)
 

In [0]:
# ── Final column order 
df_gold = gold_enriched.select(
    # Keys
    "symbol", "coin_name", "trade_date",
 
    # Price
    "open_price", "high_price", "low_price", "close_price",
    "Volume", "market_cap",
 
    # Technical Indicators
    "ma_7", "ma_14", "ma_30",
    "price_change_pct", "price_range", "volatility_7d", "ma_signal",
 
    # Sentiment (from S2)
    "avg_sentiment_score", "dominant_sentiment",
    "post_count", "avg_engagement_score",
 
    # Market Metrics (from S3)
    "active_addresses", "transaction_count",
    "avg_network_fees", "net_exchange_flow", "flow_signal",
 
    # Composite
    "bullish_score",
 
    # Metadata
    "gold_processed_at"
)
 
log.info(f"Gold rows: {df_gold.count():,}")
log.info(f"Gold columns: {len(df_gold.columns)}")
 

In [0]:
# CELL 3 — Data Quality Check

log.info("Running Gold quality checks...")
 
total         = df_gold.count()
null_symbol   = df_gold.filter(F.col("symbol").isNull()).count()
null_date     = df_gold.filter(F.col("trade_date").isNull()).count()
null_price    = df_gold.filter(F.col("close_price").isNull()).count()
neg_price     = df_gold.filter(F.col("close_price") < 0).count()
 
log.info("=" * 55)
log.info("GOLD LAYER QUALITY REPORT")
log.info("=" * 55)
log.info(f"Total rows        : {total:,}")
log.info(f"Null symbols      : {null_symbol}")
log.info(f"Null dates        : {null_date}")
log.info(f"Null close price  : {null_price}")
log.info(f"Negative prices   : {neg_price}")
 
if null_symbol == 0 and null_date == 0 and null_price == 0 and neg_price == 0:
    log.info("ALL GOLD QUALITY CHECKS PASSED")
else:
    log.warning("QUALITY ISSUES FOUND")
 
log.info("Unique symbols:")
df_gold.select("symbol").distinct().orderBy("symbol").show(30, truncate=False)
 
log.info("Bullish score distribution:")
df_gold.groupBy("bullish_score").count().orderBy("bullish_score").show()
 
log.info("Sentiment distribution:")
df_gold.groupBy("dominant_sentiment").count().orderBy("dominant_sentiment").show()
 
log.info("Flow signal distribution:")
df_gold.groupBy("flow_signal").count().orderBy("flow_signal").show()

In [0]:
# CELL 4 — Write Gold Delta (Create or Merge)
 
log.info(f"Writing Gold Delta to: {GOLD_PATH}")
 
try:
    dbutils.fs.ls(GOLD_PATH)
    path_exists = True
except Exception:
    path_exists = False
 
if path_exists and DeltaTable.isDeltaTable(spark, GOLD_PATH):
    log.info("Gold table exists — running MERGE...")
    (
        DeltaTable.forPath(spark, GOLD_PATH)
        .alias("target")
        .merge(
            df_gold.alias("source"),
            "target.symbol = source.symbol AND target.trade_date = source.trade_date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    log.info("MERGE complete")
 
else:
    log.info("Creating Gold table (first run)...")
    (
        df_gold.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("symbol")
        .save(GOLD_PATH)
    )
    log.info("Gold table created")
 

In [0]:
# CELL 5 — Verify + Preview
 
log.info("Verifying Gold Layer...")
df_verify = spark.read.format("delta").load(GOLD_PATH)
 
log.info(f"Total rows    : {df_verify.count():,}")
log.info(f"Unique symbols: {df_verify.select('symbol').distinct().count()}")
log.info(f"Date range    : {df_verify.agg(F.min('trade_date'), F.max('trade_date')).collect()[0]}")
 
df_verify.printSchema()
 
display(
    df_verify.select(
        "symbol", "trade_date", "close_price",
        "ma_7", "price_change_pct", "volatility_7d", "ma_signal",
        "avg_sentiment_score", "dominant_sentiment",
        "net_exchange_flow", "flow_signal", "bullish_score"
    ).orderBy("symbol", "trade_date").limit(20)
)

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

log.info("Building Star Schema...")

spark.sql("CREATE CATALOG IF NOT EXISTS crypto_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS crypto_catalog.gold MANAGED LOCATION 'abfss://goldlayer@cryptodl.dfs.core.windows.net/managed/'")

# ── dim_coin
dim_coin = (
    df_verify
    .groupBy("symbol")
    .agg(F.first("coin_name", ignorenulls=True).alias("coin_name"))
    .withColumn("coin_key", F.monotonically_increasing_id())
    .orderBy("symbol")
)
spark.sql("DROP TABLE IF EXISTS crypto_catalog.gold.dim_coin")
dim_coin.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto_catalog.gold.dim_coin")
log.info(f"dim_coin: {dim_coin.count()} rows")

# ── dim_date 
dim_date = (
    df_verify
    .select("trade_date").distinct()
    .withColumn("date_key",    F.monotonically_increasing_id())
    .withColumn("year",        F.year("trade_date"))
    .withColumn("quarter",     F.quarter("trade_date"))
    .withColumn("month",       F.month("trade_date"))
    .withColumn("month_name",  F.date_format("trade_date", "MMMM"))
    .withColumn("day",         F.dayofmonth("trade_date"))
    .withColumn("day_name",    F.date_format("trade_date", "EEEE"))
    .withColumn("day_of_week", F.dayofweek("trade_date"))
    .withColumn("is_weekend",  F.when(F.dayofweek("trade_date").isin(1,7), True).otherwise(False))
    .orderBy("trade_date")
)
spark.sql("DROP TABLE IF EXISTS crypto_catalog.gold.dim_date")
dim_date.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto_catalog.gold.dim_date")
log.info(f"dim_date: {dim_date.count()} rows")

# ── dim_sentiment
dim_sentiment = (
    df_verify
    .select("dominant_sentiment").distinct()
    .withColumn("sentiment_key", F.monotonically_increasing_id())
    .withColumn("sentiment_score_range",
        F.when(F.col("dominant_sentiment") == "positive", "score >= 0.05")
         .when(F.col("dominant_sentiment") == "negative", "score <= -0.05")
         .otherwise("score between -0.05 and 0.05")
    )
)
spark.sql("DROP TABLE IF EXISTS crypto_catalog.gold.dim_sentiment")
dim_sentiment.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto_catalog.gold.dim_sentiment")
log.info(f"dim_sentiment: {dim_sentiment.count()} rows")

# ── dim_flow_signal 
dim_flow_signal = (
    df_verify
    .select("flow_signal").distinct()
    .withColumn("flow_signal_key", F.monotonically_increasing_id())
    .withColumn("flow_description",
        F.when(F.col("flow_signal") == "accumulation", "More coins leaving exchanges — Bullish")
         .when(F.col("flow_signal") == "distribution", "More coins entering exchanges — Bearish")
         .otherwise("Neutral flow")
    )
)
spark.sql("DROP TABLE IF EXISTS crypto_catalog.gold.dim_flow_signal")
dim_flow_signal.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto_catalog.gold.dim_flow_signal")
log.info(f"dim_flow_signal: {dim_flow_signal.count()} rows")

# ── dim_ma_signal 
dim_ma_signal = (
    df_verify
    .select("ma_signal").distinct()
    .withColumn("ma_signal_key", F.monotonically_increasing_id())
    .withColumn("ma_description",
        F.when(F.col("ma_signal") == "above_ma", "Price above 7-day MA — Uptrend")
         .when(F.col("ma_signal") == "below_ma", "Price below 7-day MA — Downtrend")
         .otherwise("Price at 7-day MA — Sideways")
    )
)
spark.sql("DROP TABLE IF EXISTS crypto_catalog.gold.dim_ma_signal")
dim_ma_signal.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("crypto_catalog.gold.dim_ma_signal")
log.info(f"dim_ma_signal: {dim_ma_signal.count()} rows")

# ── fact_crypto_daily — MERGE 
fact_new = (
    df_verify
    .join(dim_coin,        on="symbol",            how="left")
    .join(dim_date,        on="trade_date",         how="left")
    .join(dim_sentiment,   on="dominant_sentiment", how="left")
    .join(dim_flow_signal, on="flow_signal",        how="left")
    .join(dim_ma_signal,   on="ma_signal",          how="left")
    .select(
        "coin_key", "date_key", "sentiment_key",
        "flow_signal_key", "ma_signal_key",
        "symbol", "trade_date",
        "open_price", "high_price", "low_price", "close_price",
        "Volume", "market_cap",
        "ma_7", "ma_14", "ma_30",
        "price_change_pct", "price_range", "volatility_7d",
        "avg_sentiment_score", "post_count", "avg_engagement_score",
        "active_addresses", "transaction_count",
        "avg_network_fees", "net_exchange_flow",
        "bullish_score", "gold_processed_at"
    )
)

if spark.catalog.tableExists("crypto_catalog.gold.fact_crypto_daily"):
    log.info("Fact table exists — running MERGE...")
    (
        DeltaTable.forName(spark, "crypto_catalog.gold.fact_crypto_daily")
        .alias("target")
        .merge(
            fact_new.alias("source"),
            "target.symbol = source.symbol AND target.trade_date = source.trade_date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    log.info("MERGE complete")
else:
    log.info("Creating fact table (first run)...")
    fact_new.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("crypto_catalog.gold.fact_crypto_daily")
    log.info("fact_crypto_daily created")

# ── Verify 
log.info("=" * 55)
log.info("STAR SCHEMA COMPLETE")
log.info("=" * 55)
log.info(f"fact rows     : {spark.table('crypto_catalog.gold.fact_crypto_daily').count():,}")
log.info(f"dim_coin      : {spark.table('crypto_catalog.gold.dim_coin').count()}")
log.info(f"dim_date      : {spark.table('crypto_catalog.gold.dim_date').count()}")
log.info(f"dim_sentiment : {spark.table('crypto_catalog.gold.dim_sentiment').count()}")
log.info(f"dim_flow      : {spark.table('crypto_catalog.gold.dim_flow_signal').count()}")
log.info(f"dim_ma        : {spark.table('crypto_catalog.gold.dim_ma_signal').count()}")
display(spark.sql("SHOW TABLES IN crypto_catalog.gold"))